In [ ]:
# Set knit directory
#setwd(here::here('./'))
knitr::opts_knit$set(root.dir = setwd(here::here('./')))
getwd()

In [ ]:
options(scipen = 999)

# Save figures in specific place
knitr::opts_chunk$set(autodep        = TRUE,
                      #cache.comments = TRUE,
                      comment = "",
                      # Save figures as pdf ?
                      dev = c( "png", "pdf"),
                  
                      # Include code?
                      echo           = TRUE,
                      error          = FALSE,
                      fig.align      = "center",
                      
                      # Path where figures are going to be store pdf single 
                      # figures
                      fig.path       = paste0("./figures_pca", "/"),
                      fig.width      = 16,
                      fig.height     = 10,
                      message        = FALSE,
                      warning        = FALSE)

In [ ]:

library(austraits)
library(ggplot2)
library(pcaMethods)
library(ggrepel)

# For data manipulation
library(dplyr)
library(purrr)
library(stringr)
library(tidyr)
library(lubridate)
library(here)
library(tibble)

# Get soft traits from Austraits

In [ ]:
# Read whole db
austraits <- readRDS(gzfile("./data/austraits/file12a9f3e0c74d5.download", "rb"))

In [ ]:
traits_data <- tibble::tibble(austraits$traits) 

In [ ]:
soft_traits <- 
    traits_data %>%
  
      # Species
      filter(taxon_name %in% c(# Fast growing species
                               "Eucalyptus saligna", 
                               "Eucalyptus tereticornis",
                               "Eucalyptus dunnii", 
                               "Eucalyptus grandis",
                                              
                               # Slow growing species
                               "Eucalyptus crebra", 
                               "Eucalyptus fibrosa",
                               "Eucalyptus sideroxylon", 
                               "Eucalyptus melliodora"
                               )) %>% 
  
        arrange(taxon_name) %>%  
        filter(life_stage == "adult") %>% 
        filter(value_type == "mean" | value_type == "raw") %>%
        
        # Traits  
        filter(trait_name %in% c( 
                                 "leaf_mass_per_area", 
                                 "seed_dry_mass", 
                                 "leaf_N_per_area",
                                 "leaf_dry_matter_content"))  %>% 
        
        # Cleaning
        select(taxon_name, trait_name, value, unit)  %>% 
      
        mutate(value = as.numeric(value)) %>%  
  
        group_by(taxon_name, trait_name, unit) %>%
  
        summarise(median_value = median(value), n_obs = n())

In [ ]:
reactable::reactable(soft_traits) 

In [ ]:
soft_traits_cleaned <- 
    soft_traits %>% 
    mutate(taxon_name = factor(taxon_name),
           trait_name = factor(trait_name),
           trait_value = as.numeric(median_value), 
           .keep = "unused"
           ) %>% 
    group_by(trait_name, taxon_name) %>% 
    select(-c(n_obs, unit))  
 

# Load Hard traits

In [ ]:
hard_traits <- 
  read.csv('./data/hard_traits.csv', na.strings = "NA")

In [ ]:
reactable::reactable(hard_traits)

In [ ]:
hard_traits_cleaned <- 
   hard_traits %>% 
    mutate(taxon_name = factor(taxon_name),
           trait_name = factor(trait_name),
           trait_value = as.numeric(trait_value), 
           .keep = "unused"
           ) %>% 
    group_by(trait_name, taxon_name) %>% 
    select(-c(units, source, link))

In [ ]:
glimpse(hard_traits_cleaned)

# Merge soft and hard traits

In [ ]:
data_pca <- 
    rbind(hard_traits_cleaned, soft_traits_cleaned) %>% 
    
    # Pivot data for PCA
    pivot_wider(names_from = trait_name, 
                values_from = trait_value)  %>% 
    column_to_rownames("taxon_name")

# PCA

In [ ]:

# This method was used because of the presence of NA's
(pca_results <- pca(data_pca, method = "nipals", nPcs = 7, scale = "uv"))

## Biplot

In [ ]:
# Extract scores 
scores_df <- as.data.frame(scores(pca_results))
scores_df$sample <- rownames(scores_df)

In [ ]:
# Extract loadings
loadings_df <- as.data.frame(loadings(pca_results))
loadings_df$variable <- rownames(loadings_df)

In [ ]:
# Scale loadings to scores range for overlay
scale_factor <- max(abs(scores_df[, 1:4])) / max(abs(loadings_df[, 1:4])) * 0.7

loadings_df[, 1:4] <- loadings_df[, 1:4] * scale_factor

# Get variance explained for axis labels
r2 <- round(pca_results@R2 * 100, 1)


## Clasify species into Fast or slow growing

In [ ]:
# Extract PC1, PC2, PC3, PC4 scores
pc_scores <- as.data.frame(scores(pca_results))[, 1:4]

### Run Kmeans algorithm 

In [ ]:
set.seed(42)
kmeans <- kmeans(pc_scores, centers = 2, nstart = 25)

In [ ]:
pc_scores <- 
    pc_scores %>% 
      mutate(group = as.factor(kmeans$cluster),
             species = rownames(pc_scores),
             life_history_group = if_else(species == "Eucalyptus saligna" |
                                            species == "Eucalyptus dunnii" |
                                            species == "Eucalyptus grandis" |
                                            species == "Eucalyptus tereticornis", 
                                          
                                          "fast_growing", "slow_growing")) %>%
      
      select(species, life_history_group, everything())

# Biplot

In [ ]:
trait_labels <- c("leaf_mass_per_area"      = "LMA~(g~cm^-2)",
                  "wood_density"            = "Wood~density~(g~cm^-3)",
                  "volume_index_change"     = "Growth~rate~(m^3~ha^-1~year^-1)",
                  "p50"                     = "P[50]~(MPa)",
                  "leaf_dry_matter_content" = "Leaf~dry~matter~content~(g/g)",
                  "seed_dry_mass"           = "Seed~dry~mass~(mg)",
                  "leaf_N_per_area"         = "Leaf~N~per~area~(g~cm^-2)",
                  "gs50"                    = "g[s50]~(MPa)",
                  "gsmax"                   = "g[smax]~(mol~m^-2~s^-1)")

In [ ]:
(euc_pca <- 
    ggplot() +
        
        geom_hline(yintercept = 0, linetype = "dashed", color = "#F2F2F2") +
        geom_vline(xintercept = 0, linetype = "dashed", color = "#F2F2F2") +
    
      
        geom_point(data = pc_scores, aes(x = PC1, y = PC2,
                                         color = life_history_group),
                   size = 10) +
      
        # Loadings arrows
        geom_segment(data = loadings_df, aes(x = 0, y = 0, xend = PC1, yend = PC2),
                     
                     arrow = arrow(length = unit(0.7, "cm")),
                     color = "#E5E4E2", 
                     linewidth = 3) +
      
        # Traits
        geom_text_repel(data = loadings_df, aes(x = PC1, y = PC2,
                                                label = recode(variable, !!!trait_labels)),
                        size = 15, 
                        color ="#3B3B3B", 
                        fontface = "bold",
                        parse = TRUE,
                        seed          = 42,
                        
                        # spread trait labels apart
                        box.padding   = 0.2,
                        
                        # back off the arrow tips
                        point.padding = 0.1,   
                        force         = 2,
                        max.overlaps  = Inf) +
        
        # Species names
        geom_text_repel(data = pc_scores, aes(x = PC1, y = PC2, label = species,
                                              color = life_history_group),
                        show.legend = FALSE,
                        size = 15, 
                        fontface = "italic",
                        seed          = 42,
                        
                        # spread species names apart
                        box.padding   = 4.6,   
                        
                        # keep clear of the dots
                        point.padding = 0,
                        force         = 3,
                        max.overlaps  = Inf) +
    
      
        # Axis names
        labs(x = paste0("PC1 (", r2[1], "%)"), y = paste0("PC2 (", r2[2], "%)")) +
      
        theme_bw(base_size = 60) +
            
        theme(legend.position = "inside",
              legend.title = element_blank(),
              legend.justification = c(0.005, 0.99),
              legend.position.inside = c(0.005, 0.99),
              legend.text = element_text(size = 40),
              legend.background = element_blank(),
              panel.grid.major = element_blank(),
              panel.grid.minor = element_blank(),
              strip.background = element_rect(fill = "white",colour = NA),
              strip.text = element_text(colour = "black",face = "bold"),
              panel.border = element_rect(colour = "black",fill = NA,
                                              linewidth = 4),
              axis.text    = element_text(colour = "black"),
              axis.ticks   = element_line(colour = "black")) +
    
        scale_colour_manual(values = c("fast_growing" = "#e34948",
                                       "slow_growing" = "#2a78d6"),
                            
                            breaks = c("slow_growing", "fast_growing"),
                            
                            labels = c("slow_growing" = "Slow-Safe",
                                       "fast_growing" = "Fast-Risky"))
)



In [ ]:
ggsave("../666_figures/4_supplementary_figure_1_euc_pca.png", 
       plot = euc_pca, 
       width = 25, height = 20, units = "in", dpi = 1000)